In [2]:
import pymupdf

doc = pymupdf.open("sample.pdf")
pages = []
for page_number, page in enumerate(doc, start=1):
    text = page.get_text()
    pages.append({"page_number": page_number, "text": text})

print(len(pages))
print(pages[0]["text"][:500])

43
Open Access
© The Author(s) 2023. Open Access  This article is licensed under a Creative Commons Attribution 4.0 International License, which permits 
use, sharing, adaptation, distribution and reproduction in any medium or format, as long as you give appropriate credit to the original 
author(s) and the source, provide a link to the Creative Commons licence, and indicate if changes were made. The images or other third 
party material in this article are included in the article’s Creative Common


In [7]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

CHUNK_SIZE = 800
CHUNK_OVERLAP = 100

splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
)

chunks = []
for page in pages:
    pieces = splitter.split_text(page["text"])
    for piece in pieces:
        chunks.append({"page_number": page["page_number"], "content": piece.replace("\xa0", "").replace("\n", "")})

print(len(chunks))
print(chunks[5])

232
{'page_number': 1, 'content': 'tasks, DL, mainly throughCNNs, ispreferred whenample data andcomputational resources are available andshowgood detection andclassificationeffects ontheir datasets, butnoton other datasets. Finally, inthis paper, theauthor aims tokeep future researchers up-to-date withtheperformances, evaluation metrics, andresults ofpreviously used techniques todetect andclassify different forms ofplant leaf orcrop *Correspondence:   wubetubarud@gmail.com; wubetuB@wcu.edu.et1 Department ofInformation Technology, Wachemo University, Hossana, Ethiopia'}


In [8]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

texts = [c["content"] for c in chunks]
embeddings = model.encode(texts, show_progress_bar=True)

print(embeddings.shape)

c:\Users\Bhavesh\Music\RAG Project\rag-document-assistant\backend\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\Bhavesh\Music\RAG Project\rag-document-assistant\backend\.venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Bhavesh\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need

(232, 384)


In [9]:
import os
import psycopg
from dotenv import load_dotenv
from pgvector.psycopg import register_vector

load_dotenv("../backend/.env")

conn = psycopg.connect(os.getenv("DATABASE_URL"))
register_vector(conn)

with conn.cursor() as cur:
    cur.execute(
        "INSERT INTO users (email, password_hash) VALUES (%s, %s) RETURNING id",
        ("test@example.com", "not-a-real-hash"),
    )
    user_id = cur.fetchone()[0]
    cur.execute(
        "INSERT INTO documents (user_id, file_name) VALUES (%s, %s) RETURNING id",
        (user_id, "sample.pdf"),
    )
    document_id = cur.fetchone()[0]
conn.commit()

print(user_id, document_id)

1 1


In [11]:
conn = psycopg.connect(os.getenv("DATABASE_URL"))
register_vector(conn)

rows = [
    (document_id, c["content"], c["page_number"], emb)
    for c, emb in zip(chunks, embeddings)
]

with conn.cursor() as cur:
    cur.executemany(
        "INSERT INTO chunks (document_id, content, page_number, embedding) VALUES (%s, %s, %s, %s)",
        rows,
    )
conn.commit()

with conn.cursor() as cur:
    cur.execute("SELECT count(*) FROM chunks WHERE document_id = %s", (document_id,))
    print(cur.fetchone()[0])

232
